In [ ]:
# Install required packages for RLT training
!pip install -q transformers torch datasets accelerate peft bitsandbytes
!pip install -q trl wandb einops sentencepiece
!pip install -q anthropic  # For Claude API if needed

import torch
import numpy as np
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, 
    TrainingArguments, Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset, load_dataset
from trl import SFTTrainer

print("Libraries imported successfully!")


In [ ]:
# Check GPU availability and system specs
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch version: {torch.__version__}")
else:
    print("⚠️  GPU not available. Training will be slow on CPU.")


In [ ]:
@dataclass
class RLTConfig:
    """Configuration for RLT training"""
    # Model settings
    teacher_model_name: str = "microsoft/DialoGPT-medium"  # Can upgrade to Qwen2.5-7B
    student_model_name: str = "microsoft/DialoGPT-small"
    
    # Training settings
    max_length: int = 512
    batch_size: int = 2  # Small for Colab
    learning_rate: float = 2e-4
    num_epochs: int = 1
    warmup_steps: int = 50
    
    # LoRA settings
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    
    # RLT specific
    temperature_range: Tuple[float, float] = (0.1, 0.9)
    reward_threshold_percentile: int = 75  # Top 25% of explanations

# Initialize config
config = RLTConfig()
print("Configuration loaded:")
print(f"Teacher: {config.teacher_model_name}")
print(f"Student: {config.student_model_name}")
print(f"Batch size: {config.batch_size}")
print(f"Max length: {config.max_length}")


In [ ]:
class RLTDataProcessor:
    """Data processing for RLT training"""
    
    def __init__(self):
        self.explanation_prompt = """You are an expert teacher. Given a question and its correct answer, provide a clear, step-by-step explanation that helps a student understand the reasoning process.

Question: {question}
Correct Answer: {answer}

Provide a detailed explanation following this format:
1. Understanding the problem
2. Step-by-step reasoning  
3. Final conclusion

Explanation:"""
    
    def create_teacher_prompt(self, question: str, answer: str) -> str:
        """Create prompt for teacher model to generate explanations"""
        return self.explanation_prompt.format(question=question, answer=answer)
    
    def create_student_prompt(self, question: str, explanation: str) -> str:
        """Create prompt for student model training"""
        return f"Question: {question}\n\nExplanation: {explanation}\n\nAnswer:"
    
    def load_sample_data(self) -> Dataset:
        """Create sample mathematical reasoning data for demonstration"""
        sample_problems = [
            {
                "question": "A store sells 15 apples per day and operates 6 days a week. How many apples does it sell in 4 weeks?",
                "answer": "360"
            },
            {
                "question": "If a rectangle has length 8 meters and width 5 meters, what is its area?", 
                "answer": "40 square meters"
            },
            {
                "question": "Sarah has 24 stickers. She gives away 1/3 of them to her friends. How many stickers does she have left?",
                "answer": "16"
            },
            {
                "question": "A train travels 60 miles per hour for 3 hours. How far does it travel?",
                "answer": "180 miles"
            },
            {
                "question": "If 5 notebooks cost $25, how much does one notebook cost?",
                "answer": "$5"
            }
        ]
        return Dataset.from_list(sample_problems)

# Initialize data processor and load sample data
data_processor = RLTDataProcessor()
sample_dataset = data_processor.load_sample_data()
print(f"Loaded {len(sample_dataset)} sample problems")
print("Sample problem:", sample_dataset[0])


In [ ]:
class RLTTeacher:
    """Reinforcement Learning Teacher model implementation"""
    
    def __init__(self, config: RLTConfig):
        self.config = config
        self.tokenizer = None
        self.model = None
        self.setup_model()
    
    def setup_model(self):
        """Initialize teacher model with efficient training setup"""
        print(f"Loading teacher model: {self.config.teacher_model_name}")
        
        # Quantization config for memory efficiency on Colab
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.teacher_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Load model with quantization
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.teacher_model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Prepare for LoRA training
        self.model = prepare_model_for_kbit_training(self.model)
        
        # LoRA configuration
        lora_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            target_modules=["c_attn", "c_proj"],  # DialoGPT specific
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        
        self.model = get_peft_model(self.model, lora_config)
        
        print(f"✅ Teacher model loaded with {self.model.num_parameters()} trainable parameters")
    
    def generate_explanation(self, question: str, answer: str, temperature: float = 0.7) -> str:
        """Generate explanation for a question-answer pair"""
        prompt = data_processor.create_teacher_prompt(question, answer)
        
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=self.config.max_length,
            truncation=True,
        ).to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=temperature,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the generated explanation part
        explanation = full_response.replace(prompt, "").strip()
        
        return explanation

# Initialize teacher model
print("🔄 Setting up teacher model...")
teacher = RLTTeacher(config)


In [ ]:
class RLTStudent:
    """Student model for evaluating teacher explanations"""
    
    def __init__(self, config: RLTConfig):
        self.config = config
        self.tokenizer = None
        self.model = None
        self.setup_model()
    
    def setup_model(self):
        """Initialize student model"""
        print(f"Loading student model: {self.config.student_model_name}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.student_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.student_model_name,
            device_map="auto",
            trust_remote_code=True
        )
        
        print(f"✅ Student model loaded")
    
    def calculate_understanding_score(self, question: str, explanation: str, correct_answer: str) -> float:
        """Calculate how well student understands the explanation (dense reward signal)"""
        student_prompt = data_processor.create_student_prompt(question, explanation)
        full_prompt = student_prompt + " " + correct_answer
        
        inputs = self.tokenizer(
            full_prompt,
            return_tensors="pt",
            max_length=self.config.max_length,
            truncation=True
        ).to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Calculate log probability of correct answer given explanation
            log_probs = torch.nn.functional.log_softmax(outputs.logits, dim=-1)
            
            # Tokenize just the answer part
            answer_tokens = self.tokenizer(
                correct_answer, 
                add_special_tokens=False, 
                return_tensors="pt"
            )["input_ids"][0]
            
            # Calculate average log probability for answer tokens
            if len(answer_tokens) == 0:
                return 0.0
            
            score = 0.0
            for i, token_id in enumerate(answer_tokens):
                pos = -(len(answer_tokens) - i)
                if abs(pos) < log_probs.shape[1]:
                    score += log_probs[0, pos, token_id].item()
            
            return score / len(answer_tokens)

# Initialize student model
print("🔄 Setting up student model...")
student = RLTStudent(config)


In [ ]:
class RLTTrainer:
    """Main RLT training pipeline"""
    
    def __init__(self, teacher: RLTTeacher, student: RLTStudent, config: RLTConfig):
        self.teacher = teacher
        self.student = student
        self.config = config
        self.training_data = []
        self.rewards = []
    
    def generate_training_data(self, dataset: Dataset, num_samples: int = 20):
        """Generate explanations and calculate dense rewards"""
        print(f"🔄 Generating training data for {num_samples} samples...")
        
        for i in range(min(num_samples, len(dataset))):
            example = dataset[i]
            question = example.get('question', '')
            answer = example.get('answer', '')
            
            if not question or not answer:
                continue
            
            # Generate explanation with random temperature for diversity
            temperature = np.random.uniform(*self.config.temperature_range)
            explanation = self.teacher.generate_explanation(question, answer, temperature)
            
            # Calculate understanding score (dense reward signal)
            understanding_score = self.student.calculate_understanding_score(
                question, explanation, answer
            )
            
            training_item = {
                'question': question,
                'answer': answer,
                'explanation': explanation,
                'understanding_score': understanding_score,
                'temperature': temperature
            }
            
            self.training_data.append(training_item)
            self.rewards.append(understanding_score)
            
            if (i + 1) % 5 == 0:
                avg_reward = np.mean(self.rewards[-5:])
                print(f"  Sample {i+1}/{num_samples}, Recent Avg Reward: {avg_reward:.4f}")
        
        print(f"✅ Generated {len(self.training_data)} training examples")
        print(f"📊 Average understanding score: {np.mean(self.rewards):.4f}")
        return self.training_data
    
    def prepare_high_quality_dataset(self) -> Dataset:
        """Filter and prepare high-quality explanations for training"""
        if not self.rewards:
            raise ValueError("No training data generated yet!")
        
        threshold = np.percentile(self.rewards, self.config.reward_threshold_percentile)
        print(f"📈 Reward threshold (top {100-self.config.reward_threshold_percentile}%): {threshold:.4f}")
        
        high_quality_data = [
            item for item in self.training_data 
            if item['understanding_score'] >= threshold
        ]
        
        # Format for SFT training
        training_texts = []
        for item in high_quality_data:
            prompt = data_processor.create_teacher_prompt(item['question'], item['answer'])
            full_text = prompt + item['explanation']
            training_texts.append({'text': full_text})
        
        print(f"✅ Filtered to {len(training_texts)} high-quality examples")
        return Dataset.from_list(training_texts)
    
    def train_teacher_model(self, training_dataset: Dataset):
        """Train teacher model using supervised fine-tuning"""
        training_args = TrainingArguments(
            output_dir="./rlt_teacher_results",
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=4,  # Increase effective batch size
            learning_rate=self.config.learning_rate,
            warmup_steps=self.config.warmup_steps,
            logging_steps=5,
            save_steps=20,
            optim="paged_adamw_8bit",
            fp16=True,
            report_to=None,  # Disable wandb for simplicity
            remove_unused_columns=False,
        )
        
        trainer = SFTTrainer(
            model=self.teacher.model,
            train_dataset=training_dataset,
            tokenizer=self.teacher.tokenizer,
            args=training_args,
            dataset_text_field="text",
            max_seq_length=self.config.max_length,
        )
        
        print("🚀 Starting teacher model training...")
        trainer.train()
        
        # Save the trained model
        trainer.save_model("./rlt_teacher_final")
        print("✅ Teacher training completed!")
        
        return trainer

# Initialize RLT trainer
print("🔄 Setting up RLT trainer...")
rlt_trainer = RLTTrainer(teacher, student, config)
print("✅ RLT trainer ready!")


In [ ]:
# Step 1: Generate training data with dense rewards
print("=" * 50)
print("STEP 1: Generating Training Data")
print("=" * 50)

training_data = rlt_trainer.generate_training_data(
    sample_dataset, 
    num_samples=len(sample_dataset)  # Use all sample data
)

# Step 2: Prepare high-quality dataset
print("\n" + "=" * 50)
print("STEP 2: Preparing High-Quality Dataset")
print("=" * 50)

filtered_dataset = rlt_trainer.prepare_high_quality_dataset()

# Step 3: Train teacher model
print("\n" + "=" * 50)
print("STEP 3: Training Teacher Model")  
print("=" * 50)

trainer = rlt_trainer.train_teacher_model(filtered_dataset)


In [ ]:
# Visualize training progress and rewards
def plot_rlt_metrics(rewards: List[float], training_data: List[Dict]):
    """Plot RLT training metrics and analysis"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Reward progression
    axes[0, 0].plot(rewards, 'b-', alpha=0.7)
    axes[0, 0].axhline(y=np.mean(rewards), color='r', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(rewards):.3f}')
    axes[0, 0].set_title('Understanding Scores Over Training')
    axes[0, 0].set_xlabel('Sample')
    axes[0, 0].set_ylabel('Understanding Score')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Reward distribution
    axes[0, 1].hist(rewards, bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 1].axvline(x=np.mean(rewards), color='r', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(rewards):.3f}')
    axes[0, 1].set_title('Distribution of Understanding Scores')
    axes[0, 1].set_xlabel('Understanding Score')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Temperature vs Reward correlation
    temperatures = [item['temperature'] for item in training_data]
    axes[1, 0].scatter(temperatures, rewards, alpha=0.6, c='green')
    axes[1, 0].set_title('Temperature vs Understanding Score')
    axes[1, 0].set_xlabel('Temperature')
    axes[1, 0].set_ylabel('Understanding Score')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Quality breakdown
    threshold = np.percentile(rewards, config.reward_threshold_percentile)
    high_quality = sum(1 for r in rewards if r >= threshold)
    low_quality = len(rewards) - high_quality
    
    axes[1, 1].pie([high_quality, low_quality], 
                   labels=[f'High Quality ({high_quality})', f'Low Quality ({low_quality})'],
                   autopct='%1.1f%%', 
                   colors=['lightgreen', 'lightcoral'])
    axes[1, 1].set_title(f'Quality Distribution (Threshold: {threshold:.3f})')
    
    plt.tight_layout()
    plt.show()

# Plot metrics
plot_rlt_metrics(rlt_trainer.rewards, rlt_trainer.training_data)

# Print detailed statistics
print("=" * 50)
print("RLT TRAINING STATISTICS")
print("=" * 50)
print(f"📊 Total training examples: {len(rlt_trainer.training_data)}")
print(f"📈 Average understanding score: {np.mean(rlt_trainer.rewards):.4f}")
print(f"📉 Standard deviation: {np.std(rlt_trainer.rewards):.4f}")
print(f"🔝 Best understanding score: {np.max(rlt_trainer.rewards):.4f}")
print(f"🔻 Worst understanding score: {np.min(rlt_trainer.rewards):.4f}")

# Quality distribution
threshold = np.percentile(rlt_trainer.rewards, config.reward_threshold_percentile)
high_quality_count = sum(1 for r in rlt_trainer.rewards if r >= threshold)
print(f"\n📈 High quality examples (top {100-config.reward_threshold_percentile}%): {high_quality_count}")
print(f"🎯 Reward threshold: {threshold:.4f}")


In [ ]:
def test_trained_teacher(question: str, answer: str, show_comparison: bool = True):
    """Test the trained teacher model with a new question"""
    print("=" * 60)
    print("🧪 TESTING TRAINED TEACHER MODEL")
    print("=" * 60)
    print(f"❓ Question: {question}")
    print(f"✅ Correct Answer: {answer}")
    print("\n" + "-" * 40)
    print("🤖 Generated Explanation:")
    print("-" * 40)
    
    # Generate explanation with low temperature for consistency
    explanation = teacher.generate_explanation(question, answer, temperature=0.3)
    print(explanation)
    
    # Calculate understanding score
    understanding_score = student.calculate_understanding_score(question, explanation, answer)
    print(f"\n📊 Understanding Score: {understanding_score:.4f}")
    
    # Quality assessment
    avg_score = np.mean(rlt_trainer.rewards)
    if understanding_score > avg_score:
        quality = "🟢 ABOVE AVERAGE"
    else:
        quality = "🟡 BELOW AVERAGE"
    
    print(f"📈 Quality Assessment: {quality}")
    print(f"📊 Average training score: {avg_score:.4f}")
    
    return explanation, understanding_score

# Test with sample questions
test_questions = [
    {
        "question": "A bakery sells 36 cupcakes per hour. If they operate for 8 hours, how many cupcakes do they sell in total?",
        "answer": "288"
    },
    {
        "question": "If a square has sides of length 7 meters, what is its perimeter?",
        "answer": "28 meters"
    }
]

# Test each question
results = []
for i, test_case in enumerate(test_questions):
    explanation, score = test_trained_teacher(
        test_case["question"], 
        test_case["answer"]
    )
    results.append({
        "question": test_case["question"],
        "answer": test_case["answer"], 
        "explanation": explanation,
        "score": score
    })
    
    if i < len(test_questions) - 1:
        print("\n" + "="*60 + "\n")

print("\n" + "=" * 60)
print("📊 TESTING SUMMARY")
print("=" * 60)
avg_test_score = np.mean([r["score"] for r in results])
print(f"Average test score: {avg_test_score:.4f}")
print(f"Training average: {np.mean(rlt_trainer.rewards):.4f}")
print(f"Improvement: {((avg_test_score / np.mean(rlt_trainer.rewards)) - 1) * 100:.1f}%")


In [ ]:
def train_student_with_teacher_explanations(high_quality_threshold: float = 75):
    """Train student model using high-quality teacher explanations"""
    print("🔄 Preparing student distillation dataset...")
    
    # Filter high-quality explanations
    threshold = np.percentile(rlt_trainer.rewards, high_quality_threshold)
    high_quality_explanations = [
        item for item in rlt_trainer.training_data 
        if item['understanding_score'] >= threshold
    ]
    
    # Prepare student training data
    student_training_texts = []
    for item in high_quality_explanations:
        student_prompt = data_processor.create_student_prompt(
            item['question'], 
            item['explanation']
        )
        full_text = student_prompt + item['answer']
        student_training_texts.append({'text': full_text})
    
    student_dataset = Dataset.from_list(student_training_texts)
    print(f"✅ Student dataset prepared with {len(student_dataset)} examples")
    
    # Configure student model for training
    student.model = prepare_model_for_kbit_training(student.model)
    
    student_lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=["c_attn", "c_proj"],  # Adjust for student model architecture
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    student.model = get_peft_model(student.model, student_lora_config)
    
    # Training arguments for student
    student_training_args = TrainingArguments(
        output_dir="./rlt_student_results",
        num_train_epochs=2,  # More epochs for student
        per_device_train_batch_size=config.batch_size,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        logging_steps=5,
        save_steps=20,
        optim="paged_adamw_8bit",
        fp16=True,
        report_to=None,
    )
    
    # Initialize SFT trainer for student
    student_trainer = SFTTrainer(
        model=student.model,
        train_dataset=student_dataset,
        tokenizer=student.tokenizer,
        args=student_training_args,
        dataset_text_field="text",
        max_seq_length=config.max_length,
    )
    
    print("🚀 Starting student model training...")
    student_trainer.train()
    student_trainer.save_model("./rlt_student_final")
    print("✅ Student training completed!")
    
    return student_trainer

# Option to train student model (uncomment to run)
# print("=" * 50)
# print("STUDENT DISTILLATION")
# print("=" * 50)
# student_trainer = train_student_with_teacher_explanations()

print("⚠️  Student distillation is optional and commented out to save Colab resources.")
print("🔧 Uncomment the lines above to run student distillation.")
